# EuroCropsML x OLMoEarth Benchmark

Evaluate whether OLMoEarth embeddings improve crop classification on EuroCropsML compared to classical remote sensing features.

**Experiments:**
- A: Baseline Features + LogisticRegression
- B: Baseline Features + LightGBM
- C: OLMoEarth Embeddings + LogisticRegression
- D: OLMoEarth Embeddings + LightGBM
- E: Hybrid (Features + Embeddings) + LightGBM

---
## Section 0  Environment Setup

In [1]:
import sys, os, time, gc, json, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from src.data.inspect import get_top_classes, generate_class_distribution, count_classes
from src.data.features import combined_baseline_features
from src.encoder.olmoearth import OLMoEarthEncoder
from src.models.classical import get_classifier
from src.evaluate.metrics import compute_metrics, save_metrics
from src.viz.umap_viz import plot_umap
from src.viz.confusion import plot_confusion_matrix

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

import psutil, platform
print(f"Python: {sys.version}")
print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB")
print(f"GPU: {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'}")
print(f"Device: {DEVICE}")

Python: 3.11.15 | packaged by conda-forge | (main, Jun 11 2026, 03:27:10) [MSC v.1944 64 bit (AMD64)]
RAM: 23.7 GB
GPU: CPU
Device: cpu


In [2]:
# Paths
DATA_DIR = "E:/RSGIS/OlmoEarth/Project/Data/preprocess/preprocess"
SPLIT_DIR = "E:/RSGIS/OlmoEarth/Project/Data/split/split"
WEIGHTS_PATH = "E:/RSGIS/OlmoEarth/Project/Models/V1.2_Small/weights.pth"
USE_CASE = "overlap_latvia_vs_estonia"
OUT_DIR = "E:\RSGIS\OlmoEarth\Project\Results\\small1.2"
MAX_SAMPLES = 10000

for d in ["dataset_stats", "features", "embeddings", "figures"]:
    os.makedirs(f"{OUT_DIR}/{d}", exist_ok=True)

In [3]:
import json as _json
from concurrent.futures import ThreadPoolExecutor, as_completed

CLASS_SETTINGS = {"small": 5, "medium": 10, "large": 20}

def _load_npz(args):
    filepath, cls = args
    try:
        data = np.load(filepath, allow_pickle=True)["data"]
        return data, cls
    except Exception:
        return None

def load_split_data(top_n_classes, max_samples=MAX_SAMPLES):
    """Load train/val/test splits filtered to top-N classes."""
    top_classes = get_top_classes(DATA_DIR, top_n_classes)
    class_filter = set(top_classes)
    le = LabelEncoder()
    le.fit(top_classes)

    split_file = os.path.join(SPLIT_DIR, USE_CASE, "finetune", "region_split_all.json")
    with open(split_file) as f:
        split_data = _json.load(f)

    splits = {}
    for split_key in ["train", "val", "test"]:
        filenames = split_data[split_key]
        items = []
        for fn in filenames:
            cls = fn.split("_")[-1].replace(".npz", "")
            if cls in class_filter:
                fp = os.path.join(DATA_DIR, fn)
                if os.path.exists(fp):
                    items.append((fp, cls))

        if max_samples and len(items) > max_samples:
            items = items[:max_samples]

        print(f"  {split_key}: loading {len(items)} files...")
        results = []
        with ThreadPoolExecutor(max_workers=8) as executor:
            futures = {executor.submit(_load_npz, item): item for item in items}
            for future in tqdm(as_completed(futures), total=len(items), desc=f"  {split_key}"):
                results.append(future.result())

        X_list = [r[0] for r in results]
        y_list = [r[1] for r in results]

        if X_list:
            max_T = max(x.shape[0] for x in X_list)
            C = X_list[0].shape[1]
            X = np.zeros((len(X_list), max_T, C), dtype=np.float32)
            for i, x in enumerate(X_list):
                T = min(x.shape[0], max_T)
                X[i, :T, :] = x[:T, :]
            y = le.transform(y_list).astype(np.int64)
        else:
            X, y = np.array([]), np.array([])

        splits[split_key] = (X, y)
        print(f"  {split_key}: {len(X)} samples loaded")
        del X_list, y_list, results
        gc.collect()

    return splits, top_classes, le

---
## Section 1  Dataset Loading

Three benchmark settings: Top-10, Top-20, Top-50 classes.

In [4]:
# Load primary benchmark
PRIMARY_N = 20
print(f"Loading Top-{PRIMARY_N} classes...")
t0 = time.time()
splits_20, classes_20, le_20 = load_split_data(PRIMARY_N)
X_train_20, y_train_20 = splits_20["train"]
X_val_20, y_val_20 = splits_20["val"]
X_test_20, y_test_20 = splits_20["test"]
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Train: {len(X_train_20)} | Val: {len(X_val_20)} | Test: {len(X_test_20)}")
print(f"Shape: {X_train_20.shape}")
print(f"Classes: {len(classes_20)}")

Loading Top-20 classes...
  train: loading 10000 files...


  train:   0%|          | 0/10000 [00:00<?, ?it/s]

  train: 10000 samples loaded
  val: loading 922 files...


  val:   0%|          | 0/922 [00:00<?, ?it/s]

  val: 922 samples loaded
  test: loading 10000 files...


  test:   0%|          | 0/10000 [00:00<?, ?it/s]

  test: 10000 samples loaded
Loaded in 15.4s
Train: 10000 | Val: 922 | Test: 10000
Shape: (10000, 146, 13)
Classes: 20


---
## Section 3  Baseline Features

In [5]:
print("Extracting baseline features (Top-20)...")
t0 = time.time()

datasets = {"train": X_train_20, "val": X_val_20, "test": X_test_20}
features = {}
for split, X in tqdm(datasets.items(), desc="Features"):
    features[split] = combined_baseline_features(X)
    print(f"  {split}: {features[split].shape}")

feat_train = features["train"]
feat_val = features["val"]
feat_test = features["test"]

print(f"Done in {time.time()-t0:.1f}s")

np.save(f"{OUT_DIR}/features/feat_train.npy", feat_train)
np.save(f"{OUT_DIR}/features/feat_val.npy", feat_val)
np.save(f"{OUT_DIR}/features/feat_test.npy", feat_test)
print(f"Saved to {OUT_DIR}/features/");

Extracting baseline features (Top-20)...


Features:   0%|          | 0/3 [00:00<?, ?it/s]

  train: (10000, 14)
  val: (922, 14)
  test: (10000, 14)
Done in 0.4s
Saved to E:\RSGIS\OlmoEarth\Project\Results\small1.2/features/


---
## Section 4  OLMoEarth Embedding Extraction

In [6]:
print("Extracting embeddings (Top-20)...")
t0 = time.time()

encoder = OLMoEarthEncoder(mode="local", local_weights_path=WEIGHTS_PATH, device=DEVICE)

emb_datasets = {"train": X_train_20, "val": X_val_20, "test": X_test_20}
embeddings = {}
for split, X in tqdm(emb_datasets.items(), desc="Embeddings"):
    embeddings[split] = encoder.encode(X, batch_size=32)
    print(f"  {split}: {embeddings[split].shape}")

emb_train = embeddings["train"]
emb_val = embeddings["val"]
emb_test = embeddings["test"]

print(f"Done in {time.time()-t0:.1f}s")

np.save(f"{OUT_DIR}/embeddings/emb_train.npy", emb_train)
np.save(f"{OUT_DIR}/embeddings/emb_val.npy", emb_val)
np.save(f"{OUT_DIR}/embeddings/emb_test.npy", emb_test)
np.save(f"{OUT_DIR}/embeddings/labels.npy", y_train_20)
print(f"Saved to {OUT_DIR}/embeddings/")

del X_train_20, X_val_20, X_test_20
gc.collect()

Extracting embeddings (Top-20)...


Embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

Encoding: 100%|██████████| 313/313 [04:22<00:00,  1.19it/s]


  train: (10000, 384)


Encoding: 100%|██████████| 29/29 [00:15<00:00,  1.89it/s]


  val: (922, 384)


Encoding: 100%|██████████| 313/313 [06:55<00:00,  1.33s/it]


  test: (10000, 384)
Done in 693.8s
Saved to E:\RSGIS\OlmoEarth\Project\Results\small1.2/embeddings/


37

## Section 5  Embedding Diagnostics

In [7]:
print("Embedding Statistics (Train):")
print(f"  Shape:  {emb_train.shape}")
print(f"  Mean:   {emb_train.mean():.4f}")
print(f"  Std:    {emb_train.std():.4f}")
print(f"  Min:    {emb_train.min():.4f}")
print(f"  Max:    {emb_train.max():.4f}")
print(f"  Finite: {np.isfinite(emb_train).all()}")
print(f"  Unique (first 10): {np.unique(emb_train[:10], axis=0).shape[0]}/10")

Embedding Statistics (Train):
  Shape:  (10000, 384)
  Mean:   0.0016
  Std:    0.1001
  Min:    -0.4027
  Max:    0.3638
  Finite: True
  Unique (first 10): 10/10


In [8]:
print("Generating UMAP...")
plot_umap(
    emb_train, y_train_20,
    title="OLMoEarth Embeddings - Top 20 Classes",
    save_path=f"{OUT_DIR}/figures/umap.png"
)
print(f"Saved to {OUT_DIR}/figures/umap.png")

Generating UMAP...
Saved to E:\RSGIS\OlmoEarth\Project\Results\small1.2/figures/umap.png


---
## Section 6  Classification Benchmark

In [9]:
def run_experiment(name, X_tr, y_tr, X_te, y_te, clf_name, labels):
    """Run one experiment: scale -> train -> evaluate."""
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    clf = get_classifier(clf_name, seed=SEED)
    clf.fit(X_tr_s, y_tr)
    y_pred = clf.predict(X_te_s)
    m = compute_metrics(y_te, y_pred, labels=labels)
    m["method"] = name
    return m, y_pred

labels = sorted(set(y_test_20))
results = []
predictions = {}

In [10]:
labels = sorted(set(y_test_20))
results = []
predictions = {}

experiments = [
    ("Baseline + LogReg", feat_train, feat_test, "logreg"),
    ("Baseline + LightGBM", feat_train, feat_test, "lgbm"),
    ("Embeddings + LogReg", emb_train, emb_test, "logreg"),
    ("Embeddings + LightGBM", emb_train, emb_test, "lgbm"),
    ("Hybrid + LightGBM",
     np.concatenate([feat_train, emb_train], axis=1),
     np.concatenate([feat_test, emb_test], axis=1),
     "lgbm"),
]

for name, Xtr, Xte, clf_name in experiments:
    m, y_pred = run_experiment(name, Xtr, y_train_20, Xte, y_test_20, clf_name, labels)
    results.append(m)
    predictions[name] = y_pred
    print(f"  {name}: acc={m['overall_accuracy']:.3f} f1={m['macro_f1']:.3f}")

  Baseline + LogReg: acc=0.614 f1=0.188
  Baseline + LightGBM: acc=0.287 f1=0.056
  Embeddings + LogReg: acc=0.670 f1=0.246
  Embeddings + LightGBM: acc=0.592 f1=0.188
  Hybrid + LightGBM: acc=0.495 f1=0.062


In [11]:
def run_scaling_benchmark(top_n):
    """Run key experiments for a given number of classes."""
    splits, classes, le = load_split_data(top_n)
    X_tr, y_tr = splits["train"]
    X_te, y_te = splits["test"]
    labels = sorted(set(y_te))

    # Features
    print("  Extracting features...")
    feat_tr = combined_baseline_features(X_tr)
    feat_te = combined_baseline_features(X_te)

    # Embeddings
    print("  Extracting embeddings...")
    enc = OLMoEarthEncoder(mode="local", local_weights_path=WEIGHTS_PATH, device=DEVICE)
    emb_tr = enc.encode(X_tr, batch_size=32)
    emb_te = enc.encode(X_te, batch_size=32)
    del enc; gc.collect()

    scaling_results = []
    for name, Xtr, Xte in tqdm([
        ("Baseline + LogReg", feat_tr, feat_te),
        ("Baseline + LightGBM", feat_tr, feat_te),
        ("Embeddings + LogReg", emb_tr, emb_te),
        ("Embeddings + LightGBM", emb_tr, emb_te),
        ("Hybrid + LightGBM", np.concatenate([feat_tr, emb_tr], axis=1),
                                np.concatenate([feat_te, emb_te], axis=1)),
    ], desc=f"  Top-{top_n}"):
        clf_name = "logreg" if "LogReg" in name else "lgbm"
        m, _ = run_experiment(name, Xtr, y_tr, Xte, y_te, clf_name, labels)
        m["n_classes"] = top_n
        scaling_results.append(m)

    del splits, X_tr, X_te, feat_tr, feat_te, emb_tr, emb_te
    gc.collect()
    return scaling_results

In [12]:
import pandas as pd

all_scaling = []
for n in tqdm([10, 20, 50], desc="Scaling benchmark"):
    print(f"\n--- Top-{n} Classes ---")
    t0 = time.time()
    res = run_scaling_benchmark(n)
    all_scaling.extend(res)
    print(f"Done in {time.time()-t0:.1f}s")

df_scale = pd.DataFrame(all_scaling)
df_scale = df_scale[["n_classes", "method", "overall_accuracy", "macro_f1", "kappa"]]
df_scale.columns = ["Classes", "Method", "Accuracy", "Macro F1", "Kappa"]
print("\n" + df_scale.to_string(index=False))
df_scale.to_csv(f"{OUT_DIR}/scaling_analysis.csv", index=False)

Scaling benchmark:   0%|          | 0/3 [00:00<?, ?it/s]


--- Top-10 Classes ---
  train: loading 10000 files...


  train:   0%|          | 0/10000 [00:00<?, ?it/s]

  train: 10000 samples loaded
  val: loading 735 files...


  val:   0%|          | 0/735 [00:00<?, ?it/s]

...

  test:   0%|          | 0/10000 [00:00<?, ?it/s]

  test: 10000 samples loaded
  Extracting features...
  Extracting embeddings...


Encoding: 100%|██████████| 313/313 [06:51<00:00,  1.31s/it]


  Top-50:   0%|          | 0/5 [00:00<?, ?it/s]

Done in 759.1s

 Classes                Method  Accuracy  Macro F1     Kappa
      10     Baseline + LogReg    0.6987  0.315160  0.491467
      10   Baseline + LightGBM    0.6543  0.196816  0.266053
      10   Embeddings + LogReg    0.8150  0.367718  0.644966
      10 Embeddings + LightGBM    0.7097  0.233858  0.361709
      10     Hybrid + LightGBM    0.7211  0.260232  0.400161
      20     Baseline + LogReg    0.6139  0.188457  0.419713
      20   Baseline + LightGBM    0.2597  0.046146 -0.039912
      20   Embeddings + LogReg    0.6821  0.278289  0.502346
      20 Embeddings + LightGBM    0.1468  0.055292  0.046501
      20     Hybrid + LightGBM    0.2035  0.086158  0.095229
      50     Baseline + LogReg    0.5714  0.083011  0.381234
      50   Baseline + LightGBM    0.4753  0.023931  0.056943
      50   Embeddings + LogReg    0.6222  0.113652  0.431976
      50 Embeddings + LightGBM    0.4668  0.019226  0.006660
      50     Hybrid + LightGBM    0.0587  0.009618  0.016809


In [13]:
df = pd.DataFrame(all_scaling)
df = df[["n_classes", "method", "overall_accuracy", "macro_f1", "kappa"]].copy()
df.columns = ["Classes", "Method", "Accuracy", "Macro F1", "Kappa"]
df = df.sort_values("Macro F1", ascending=False).reset_index(drop=True)

print("=" * 70)
print("BENCHMARK RESULTS (All Classes)")
print("=" * 70)
print(df.to_string(index=False))
print("=" * 70)

df.to_csv(f"{OUT_DIR}/comparison_table.csv", index=False)
print(f"Saved to {OUT_DIR}/")

BENCHMARK RESULTS (All Classes)
 Classes                Method  Accuracy  Macro F1     Kappa
      10   Embeddings + LogReg    0.8150  0.367718  0.644966
      10     Baseline + LogReg    0.6987  0.315160  0.491467
      20   Embeddings + LogReg    0.6821  0.278289  0.502346
      10     Hybrid + LightGBM    0.7211  0.260232  0.400161
      10 Embeddings + LightGBM    0.7097  0.233858  0.361709
      10   Baseline + LightGBM    0.6543  0.196816  0.266053
      20     Baseline + LogReg    0.6139  0.188457  0.419713
      50   Embeddings + LogReg    0.6222  0.113652  0.431976
      20     Hybrid + LightGBM    0.2035  0.086158  0.095229
      50     Baseline + LogReg    0.5714  0.083011  0.381234
      20 Embeddings + LightGBM    0.1468  0.055292  0.046501
      20   Baseline + LightGBM    0.2597  0.046146 -0.039912
      50   Baseline + LightGBM    0.4753  0.023931  0.056943
      50 Embeddings + LightGBM    0.4668  0.019226  0.006660
      50     Hybrid + LightGBM    0.0587  0.009618  0

---
## Section 8  Confusion Matrix (Best Model)

In [14]:
best_method = df.iloc[0]["Method"]
best_pred = predictions[best_method]
print(f"Best model: {best_method}")

cm = confusion_matrix(y_test_20, best_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            xticklabels=classes_20, yticklabels=classes_20)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix  {best_method}')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figures/confusion_matrix.png", dpi=150)
plt.show()
print(f"Saved to {OUT_DIR}/figures/confusion_matrix.png")

Best model: Embeddings + LogReg
Saved to E:\RSGIS\OlmoEarth\Project\Results\small1.2/figures/confusion_matrix.png


---
## Section 9  Scaling Analysis

Run top experiments across Top-10, Top-20, Top-50 classes.

In [15]:
# Scaling plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
methods = df_scale["Method"].unique()
for method in methods:
    sub = df_scale[df_scale["Method"] == method]
    axes[0].plot(sub["Classes"], sub["Accuracy"], "o-", label=method)
    axes[1].plot(sub["Classes"], sub["Macro F1"], "o-", label=method)
axes[0].set_title("Accuracy vs Number of Classes")
axes[0].set_xlabel("Top-N Classes")
axes[0].set_ylabel("Accuracy")
axes[0].legend(fontsize=7)
axes[1].set_title("Macro F1 vs Number of Classes")
axes[1].set_xlabel("Top-N Classes")
axes[1].set_ylabel("Macro F1")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figures/scaling_analysis.png", dpi=150)
plt.show()

---
## Section 10  Final Interpretation

In [16]:
best = df.iloc[0]
print("=" * 70)
print("FINAL INTERPRETATION")
print("=" * 70)
print(f"\nBest method: {best['Method']}")
print(f"  Accuracy:  {best['Accuracy']:.3f}")
print(f"  Macro F1:  {best['Macro F1']:.3f}")
print(f"  Kappa:     {best['Kappa']:.3f}")

baseline_best = df[df["Method"].str.startswith("Baseline")]["Macro F1"].max()
embed_best = df[df["Method"].str.startswith("Embeddings")]["Macro F1"].max()
hybrid_f1 = df[df["Method"].str.startswith("Hybrid")]["Macro F1"].max()

print(f"\n1. Do OLMoEarth embeddings outperform classical features?")
if embed_best > baseline_best:
    print(f"   YES  Embeddings ({embed_best:.3f}) > Baseline ({baseline_best:.3f})")
else:
    print(f"   NO   Embeddings ({embed_best:.3f}) <= Baseline ({baseline_best:.3f})")

print(f"\n2. Does hybrid improve over both?")
if hybrid_f1 > max(baseline_best, embed_best):
    print(f"   YES  Hybrid ({hybrid_f1:.3f}) > max(Baseline, Embeddings) ({max(baseline_best, embed_best):.3f})")
else:
    print(f"   NO   Hybrid ({hybrid_f1:.3f}) <= max ({max(baseline_best, embed_best):.3f})")

print(f"\n3. Scaling stability:")
if len(df_scale) > 0:
    for method in ["Baseline + LightGBM", "Embeddings + LightGBM", "Hybrid + LightGBM"]:
        sub = df_scale[df_scale["Method"] == method]
        if len(sub) > 1:
            f1_range = sub["Macro F1"].max() - sub["Macro F1"].min()
            print(f"   {method}: F1 range = {f1_range:.3f}")

print(f"\n4. Is OLMoEarth useful for EuroCropsML?")
if embed_best > 0.33 or hybrid_f1 > baseline_best:
    print("   YES  Embeddings provide useful representations")
else:
    print("   NOT YET  Need more data or better tuning")

print("=" * 70)

FINAL INTERPRETATION

Best method: Embeddings + LogReg
  Accuracy:  0.815
  Macro F1:  0.368
  Kappa:     0.645

1. Do OLMoEarth embeddings outperform classical features?
   YES  Embeddings (0.368) > Baseline (0.315)

2. Does hybrid improve over both?
   NO   Hybrid (0.260) <= max (0.368)

3. Scaling stability:
   Baseline + LightGBM: F1 range = 0.173
   Embeddings + LightGBM: F1 range = 0.215
   Hybrid + LightGBM: F1 range = 0.251

4. Is OLMoEarth useful for EuroCropsML?
   YES  Embeddings provide useful representations


In [17]:
print(f"Benchmark complete. All results saved to {OUT_DIR}/")

Benchmark complete. All results saved to E:\RSGIS\OlmoEarth\Project\Results\small1.2/
